# Settings

## Import Packages

In [24]:
import osmnx as ox
import pandas as pd
import numpy as np
import os
import re
from scipy import stats
from scipy.stats import chi2_contingency, pointbiserialr

import json
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
import math
import itertools
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from shapely.geometry import Point
import requests
import time
from geopy.geocoders import Nominatim
import time
from shapely.wkt import dumps
from shapely.wkt import loads
from dotenv import load_dotenv

pd.set_option('display.precision', 4)

## Self-defined functions

### extract_main_location

In [2]:
def extract_main_location(text):
    match_colon = re.match(r"^(.*?):", text)
    if match_colon:
        return match_colon.group(1).strip()

    match_between = re.match(r"^(.*?)\s+between\s+", text, re.IGNORECASE)
    if match_between:
        return match_between.group(1).strip()

    return text

### save_geoDataFrame

In [13]:
def save_geoDataFrame(source_df, filename):
    """
    Save a GeoDataFrame to a CSV file by converting the geometry column to WKT format.

    Parameters:
        source_df (GeoDataFrame): The GeoDataFrame to be saved.
        filename (str): Path to the output CSV file.

    Notes:
        - The geometry column will be converted to WKT strings using shapely.wkt.dumps().
        - The resulting CSV can be restored into a GeoDataFrame using shapely.wkt.loads().
    """
    df = source_df.copy()
    df['geometry'] = df['geometry'].apply(dumps)
    df.to_csv(filename, index=False)

### get_event_geometry

In [4]:
def get_event_geometry(location_name, buffer_radius_m=200, city_prefix="Manhattan, New York, NY", pause=1.0):
    """
    Get geometry for a location using polygon (OSM) or buffered point (fallback).

    Parameters:
        location_name (str): The name of the event/location.
        buffer_radius_m (float): Fallback buffer radius in meters.
        city_prefix (str): Prefix to help disambiguate location.
        pause (float): Seconds to sleep between geocoding queries.

    Returns:
        geometry (shapely.geometry): Polygon or buffered point.
        source (str): 'polygon' if from OSM, 'buffer' if fallback point used, or 'failed'.
    """
    geolocator = Nominatim(user_agent="geo_fallback")
    full_location = f"{location_name}, {city_prefix}"

    # Try OSMnx polygon
    try:
        gdf = ox.geocode_to_gdf(full_location)
        if not gdf.empty and gdf.geometry.iloc[0].geom_type in ['Polygon', 'MultiPolygon']:
            return gdf.geometry.iloc[0], 'polygon'
    except Exception as e:
        pass

    # Pause to respect geocoding API limits
    time.sleep(pause)

    # Fallback: geopy + buffer
    try:
        loc = geolocator.geocode(full_location)
        if loc:
            point = Point(loc.longitude, loc.latitude)
            buffer_radius_deg = buffer_radius_m / 111000  # ~1 deg = 111km
            return point.buffer(buffer_radius_deg), 'buffer'
    except Exception as e:
        pass

    return None, 'failed'

### standardize_address

In [ ]:
def standardize_address(text):
    text = text.upper()
    text = text.replace('WEST', 'W')
    text = text.replace('EAST', 'E')
    text = text.replace('STREET', 'St')
    text = text.replace('AVENUE', 'Ave')
    text = text.replace(' BOULEVARD', ' Blvd')
    text = text.replace(' PLACE', ' Pl')
    text = text.replace(' ROAD', ' Rd')
    text = text.replace(' CIRCLE', ' Cir')
    return text

### fill_missing_geometry_with_google_api

In [26]:
def fill_missing_geometry_with_google_api(location_df,
                                        location_col='fixed_main_location',
                                        api_key=None,
                                        buffer_radius_m=200,
                                        sleep_time=1.0):
    """
    Use Google Maps API to fill in missing geometry with a buffered point.

    Parameters:
        location_df (pd.DataFrame): DataFrame with a 'fixed_main_location' column.
        location_col (str): Column with standardized location names.
        api_key (str): Google Maps API key.
        buffer_radius_m (float): Buffer radius in meters.
        sleep_time (float): Seconds to pause between API calls.

    Returns:
        pd.DataFrame: Updated DataFrame with 'geometry' and 'source_type'.
    """
    assert api_key, "Google Maps API key (api_key) is required."
    buffer_radius_deg = buffer_radius_m / 111000  # ~1 degree ≈ 111 km

    for i, row in location_df[location_df['geometry'].isna()].iterrows():
        try:
            query = f"{row[location_col]}, Manhattan, New York, NY"
            url = f"https://maps.googleapis.com/maps/api/geocode/json?address={query}&key={api_key}"
            response = requests.get(url).json()

            if response['status'] == 'OK' and response['results']:
                location = response['results'][0]['geometry']['location']
                lat, lon = location['lat'], location['lng']
                point = Point(lon, lat)
                location_df.at[i, 'geometry'] = point.buffer(buffer_radius_deg)
                location_df.at[i, 'source_type'] = 'google'

                print(f"{i}: {query} → ({lat}, {lon})")
            else:
                print(f"❌ Could not resolve location: {query}")
        except Exception as e:
            print(f"⚠️ Error @ {row[location_col]}: {e}")

        time.sleep(sleep_time)

    return location_df

# NYC Permitted Event Information
- Instruction:
  - Download from [NYC Permitted Event Information](https://data.cityofnewyork.us/City-Government/NYC-Permitted-Event-Information-Historical/bkfu-528j/explore/query/SELECT%0A%20%20%60start_date_time%60%2C%0A%20%20%60end_date_time%60%2C%0A%20%20%60event_name%60%2C%0A%20%20%60event_type%60%2C%0A%20%20%60event_location%60%2C%0A%20%20%60event_id%60%2C%0A%20%20count%28%60event_id%60%29%20AS%20%60records%60%0AWHERE%0A%20%20caseless_eq%28%60event_borough%60%2C%20%22Manhattan%22%29%0A%20%20AND%20%28%60start_date_time%60%20%3E%3D%20%222024-04-01T00%3A00%3A00%22%20%3A%3A%20floating_timestamp%29%0AGROUP%20BY%0A%20%20%60start_date_time%60%2C%0A%20%20%60end_date_time%60%2C%0A%20%20%60event_name%60%2C%0A%20%20%60event_type%60%2C%0A%20%20%60event_location%60%2C%0A%20%20%60event_id%60%0AORDER%20BY%20%60start_date_time%60%20ASC%20NULL%20LAST%2C%20%60end_date_time%60%20ASC%20NULL%20LAST/page/column_manager)
    - Filters applied:
      1. `Event Borough` = Manhattan
      2. `Start Date/Time` >= 2024 Apr 01 12:00:00 AM
    - Grouped by `Start Date/Time`, `End Date/Time`, `Event Name`, `Event Type`, `Event Location`, `Event ID`
  - Save the exported result as `event_data.csv` into `raw_data/` directory
- The processed event data has been saved to the [shared Google Drive folder](https://drive.google.com/drive/u/2/folders/1Yr7l0EcolHcL2TDrrq72ZUix2FNEc5X2)

In [3]:
event_df = pd.read_csv('raw_data/event_data.csv')
event_df.rename(columns = {'Event ID': 'event_id', 
                           'Event Name': 'event_name', 
                           'Start Date/Time': 'start_time',
                           'End Date/Time': 'end_time',
                           'Event Location': 'event_location'}, inplace=True
                           )
event_df['start_time'] = pd.to_datetime(event_df['start_time'],format='%m/%d/%Y %I:%M:%S %p')
event_df['end_time'] = pd.to_datetime(event_df['end_time'],format='%m/%d/%Y %I:%M:%S %p')

# Select events between 2024-04-01 and 2025-03-31 (inclusive)
event_df = event_df[(event_df['start_time'] <= pd.to_datetime('2025-03-31')) & (event_df['end_time'] >= pd.to_datetime('2024-04-01'))]
event_df['main_location'] = event_df['event_location'].apply(extract_main_location)
event_df.head()

,start_time,end_time,event_name,Event Type,event_location,event_id,main_location
0,2024-04-01 00:01:00,2024-04-15 06:00:00,Permitted Film Event,Theater Load in and Load Outs,"20 LINCOLN CENTER PLAZA, WEST 62 STREET betw...",765889,"20 LINCOLN CENTER PLAZA, WEST 62 STREET"
1,2024-04-01 00:01:00,2024-04-15 06:00:00,Permitted Film Event,Theater Load in and Load Outs,"20 LINCOLN CENTER PLAZA, WEST 62 STREET betw...",768867,"20 LINCOLN CENTER PLAZA, WEST 62 STREET"
2,2024-04-01 05:00:00,2024-04-15 22:00:00,Permitted Film Event,Theater Load in and Load Outs,Studio 54: 254 West 54th Street,771542,Studio 54
3,2024-04-01 06:00:00,2024-04-30 23:00:00,Permitted Film Event,Theater Load in and Load Outs,American Airlines Theatre: 227 West 42nd Street,771973,American Airlines Theatre
4,2024-04-01 16:00:00,2024-04-01 18:00:00,Celebration,Special Event,Central Park: Cop Cot,738613,Central Park


## Geocoding Event Locations
To associate each event location with a geographic area in Manhattan, we adopted a multi-step geocoding strategy that combines OpenStreetMap and Google Maps data to ensure both precision and completeness:

**1. Initial OSM Geocoding (Polygon Preferred, Buffer as Fallback):**

We first attempted to retrieve polygon geometries (e.g., park or district boundaries) for each `main_location` using `osmnx.geocode_to_gdf()`. If a polygon was available, it was used directly. If only a point was returned via `geolocator.geocode`, a 200-meter buffer was applied around the point to approximate the spatial extent of the event.

**2. Standardized Address and Retry:**

For locations that failed in the initial attempt, we applied a custom `standardize_address()` function to clean and normalize the location name. We then retried the same OSM-based geocoding logic using the standardized address via the `get_event_geometry()` function.

**3. Final Fallback Using Google Maps API:**

If both OSM methods failed to resolve the location, we used the Google Maps Geocoding API to retrieve latitude and longitude. A 200-meter buffer was again applied to generate an approximate polygonal area.

**4. Manual Address Correction and Retry:**

If all automated methods failed, we manually verified and corrected the location name using Google Maps. We then repeated the geocoding attempts (OSM polygon or buffered point, followed by Google API) to generate a usable geometry.

This hierarchical geocoding approach helps ensure broad coverage of event locations while preserving spatial accuracy for modeling and analysis.

### 1. Initial OSM Geocoding (Polygon Preferred, Buffer as Fallback)

In [6]:
location_df = event_df[['main_location']].drop_duplicates().reset_index(drop=True)
location_df['geometry'] = None
location_df['source_type'] = None

for i, row in location_df.iterrows():
    geom, src = get_event_geometry(row['main_location'])
    location_df.at[i, 'geometry'] = geom
    location_df.at[i, 'source_type'] = src
    print(i, row['main_location'], geom, src)

0 20 LINCOLN CENTER PLAZA, WEST   62 STREET None failed
1 Studio 54 POLYGON ((-73.9839362 40.7642717, -73.9838941 40.7642539, -73.9839108 40.7642311, -73.9838423 40.7642021, -73.9838257 40.7642249, -73.9837791 40.7642051, -73.9836212 40.7644205, -73.9836952 40.7644529, -73.9837779 40.7644868, -73.9839362 40.7642717)) polygon
2 American Airlines Theatre POLYGON ((-73.9883637 40.7570979, -73.9881124 40.7569912, -73.9880555 40.7569671, -73.9878748 40.7572134, -73.988183 40.7573441, -73.9883637 40.7570979)) polygon
3 Central Park POLYGON ((-73.9814075 40.7684558, -73.9813532 40.7684005, -73.9812954 40.7683388, -73.9812666 40.7682979, -73.9812424 40.7682578, -73.981225 40.7682236, -73.9812054 40.7681647, -73.9811983 40.7681234, -73.9811958 40.7680759, -73.9811947 40.7680529, -73.9811958 40.7680234, -73.9812024 40.7679925, -73.9812166 40.7679435, -73.9812094 40.7679176, -73.9811958 40.7678954, -73.9811787 40.7678877, -73.9810502 40.7678317, -73.9791766 40.7670323, -73.9763836 40.7658542, -73

### 2. Standardized Address and Retry

In [19]:
# Standardize address format and fetch geometry for records missing spatial information
location_df['fixed_main_location'] = location_df['main_location'].apply(standardize_address)
for i, row in location_df[location_df['geometry'].isna()].iterrows():
    geom, src = get_event_geometry(row['fixed_main_location'])
    location_df.at[i, 'geometry'] = geom
    location_df.at[i, 'source_type'] = src
    print(i, row['fixed_main_location'], geom, src)

0 20 LINCOLN CENTER PLAZA, W   62 St None failed
19 W   26 St None failed
21 35/36 BROADWAY PEDESTRIAN PLAZA (HERALD SQUARE) None failed
23 W   35 St None failed
28 46/47 BROADWAY PEDESTRIAN PLAZA (TIMES SQ) None failed
31 E  125 St None failed
44 130  LIBERTY St, BROADWAY BETWEEN LIBERTY St AND BEAVER St,  LIBERTY St BETWEEN GREENWICH St AND BROADWAY, BOWLING GREEN None failed
47 GANSEVOORT/13 GANSEVOORT PEDESTRIAN PLAZA None failed
54 NAGLE Ave BETWEEN BROADWAY AND ELLWOOD St,  NAGLE Ave BETWEEN ELLWOOD St AND DYCKMAN St,  DYCKMAN St BETWEEN NAGLE Ave AND BROADWAY,  BROADWAY BETWEEN DYCKMAN St AND W  218 St,  W  218 St BETWEEN BROADWAY AND INDIAN Rd, INWOOD HILL PARK None failed
65 30 LINCOLN CENTER PLAZA, AMSTERDAM Ave None failed
66 31 E   17 St None failed
77 W   23 St None failed
78 SUTTON Pl PARK None failed
85 1 ELDRIDGE St,1  ELDRIDGE St, DIVISION St None failed
97 143 E   17 St, E   17 St None failed
100 THOMAS PAINE PARK (FOLEY SQUARE) None failed
105 PALACE THEATRE None fai

In [20]:
save_geoDataFrame(location_df, 'event_data_20240401_20250331.csv')

### 3. Final Fallback Using Google Maps API

In [21]:
location_df[location_df['geometry'].isna()]

,main_location,geometry,source_type,fixed_main_location
0,"20 LINCOLN CENTER PLAZA, WEST 62 STREET",None,failed,"20 LINCOLN CENTER PLAZA, W 62 St"
19,WEST 26 STREET,None,failed,W 26 St
21,35/36 Broadway Pedestrian Plaza (Herald Square),None,failed,35/36 BROADWAY PEDESTRIAN PLAZA (HERALD SQUARE)
23,WEST 35 STREET,None,failed,W 35 St
28,46/47 Broadway Pedestrian Plaza (Times Sq),None,failed,46/47 BROADWAY PEDESTRIAN PLAZA (TIMES SQ)
...,...,...,...,...
741,235 WEST 50 STREET,None,failed,235 W 50 St
745,"611 WEST 204 STREET, WEST 204 STREET",None,failed,"611 W 204 St, W 204 St"
748,"431 WEST 16 STREET, WEST 16 STREET",None,failed,"431 W 16 St, W 16 St"
749,12/13 Gansevoort Pedestrian Plaza,None,failed,12/13 GANSEVOORT PEDESTRIAN PLAZA


In [27]:
load_dotenv()
api_key = os.getenv("GOOGLE_API_KEY")
fixed_location_df = fill_missing_geometry_with_google_api(location_df=location_df,
                                                          location_col='fixed_main_location',
                                                          api_key=api_key)
fixed_location_df

0: 20 LINCOLN CENTER PLAZA, W   62 St, Manhattan, New York, NY → (40.7717692, -73.98379969999999)
19: W   26 St, Manhattan, New York, NY → (40.7476792, -73.9982087)
21: 35/36 BROADWAY PEDESTRIAN PLAZA (HERALD SQUARE), Manhattan, New York, NY → (40.8260566, -73.9502028)
23: W   35 St, Manhattan, New York, NY → (40.749118, -73.9840802)
28: 46/47 BROADWAY PEDESTRIAN PLAZA (TIMES SQ), Manhattan, New York, NY → (40.8606211, -73.9308903)
31: E  125 St, Manhattan, New York, NY → (40.8040942, -73.9367337)
44: 130  LIBERTY St, BROADWAY BETWEEN LIBERTY St AND BEAVER St,  LIBERTY St BETWEEN GREENWICH St AND BROADWAY, BOWLING GREEN, Manhattan, New York, NY → (40.71109300000001, -74.01426400000001)
47: GANSEVOORT/13 GANSEVOORT PEDESTRIAN PLAZA, Manhattan, New York, NY → (40.7395316, -74.00499080000002)
54: NAGLE Ave BETWEEN BROADWAY AND ELLWOOD St,  NAGLE Ave BETWEEN ELLWOOD St AND DYCKMAN St,  DYCKMAN St BETWEEN NAGLE Ave AND BROADWAY,  BROADWAY BETWEEN DYCKMAN St AND W  218 St,  W  218 St BETWEEN

,main_location,geometry,source_type,fixed_main_location
0,"20 LINCOLN CENTER PLAZA, WEST 62 STREET","POLYGON ((-73.98199789819819 40.7717692, -73.9...",google,"20 LINCOLN CENTER PLAZA, W 62 St"
1,Studio 54,"POLYGON ((-73.9839362 40.7642717, -73.9838941 ...",polygon,STUDIO 54
2,American Airlines Theatre,"POLYGON ((-73.9883637 40.7570979, -73.9881124 ...",polygon,AMERICAN AIRLINES THEATRE
3,Central Park,"POLYGON ((-73.9814075 40.7684558, -73.9813532 ...",polygon,CENTRAL PARK
4,WALL STREET,"POLYGON ((-74.0050502981982 40.7047148, -74.00...",buffer,WALL St
...,...,...,...,...
748,"431 WEST 16 STREET, WEST 16 STREET",POLYGON ((-74.00392729819819 40.74308690000001...,google,"431 W 16 St, W 16 St"
749,12/13 Gansevoort Pedestrian Plaza,"POLYGON ((-74.0029314981982 40.7392749, -74.00...",google,12/13 GANSEVOORT PEDESTRIAN PLAZA
750,WASHINGTON PLACE,"POLYGON ((-73.99272419819819 40.7293955, -73.9...",buffer,WASHINGTON Pl
751,Samuel J. Friedman Theatre,"POLYGON ((-73.9869495 40.7602867, -73.9868232 ...",polygon,SAMUEL J. FRIEDMAN THEATRE


### 4. Manual Address Correction and Retry

In [29]:
# Manually fix the address
fixed_location_df.loc[fixed_location_df['geometry'].isna(), 'fixed_main_location'] = '173 E 3rd St'
fixed_location_df[fixed_location_df['geometry'].isna()]

,main_location,geometry,source_type,fixed_main_location
715,"173 EAST 3 STREET,173 EAST 3 STREET,173 ...",None,failed,173 E 3rd St


In [31]:
for i, row in fixed_location_df[fixed_location_df['geometry'].isna()].iterrows():
    geom, src = get_event_geometry(row['fixed_main_location'])
    fixed_location_df.at[i, 'geometry'] = geom
    fixed_location_df.at[i, 'source_type'] = src
    print(i, row['fixed_main_location'], geom, src)

715 173 E 3rd St None failed


In [33]:
final_location_df = fill_missing_geometry_with_google_api(location_df=fixed_location_df,
                                                          location_col='fixed_main_location',
                                                          api_key=api_key)

## Combine the Event and Geographic Information

In [39]:
merged_event_df = event_df.merge(final_location_df, on='main_location')
save_geoDataFrame(merged_event_df, 'event_data_20240401_20250331.csv')

In [41]:
save_geoDataFrame(final_location_df, 'location_data_20240401_20250331.csv')